# Effective Resistance R_eff vs N and a (distance-dependent model)

How does the network's effective resistance $R_{eff}$ (using the distance-dependent resistance model, $R_{edge} = R_0 \cdot L_{edge}$, from `distance_dependent_resistance.ipynb`) behave as a function of:

- **N** — number of sticks
- **a** — aspect ratio (stick length)

For each (N, a) pair we draw random networks until we get several that percolate (not every draw does), compute $R_{eff}$ for each with `distance_dependent=True`, and summarize the distribution with box plots.

This reuses `generate_percolating_network()` and `distance_dependent_impedance()`, now modularized into `utils/pipeline_utils.py` (previously duplicated ad hoc in `distance_dependent_resistance.ipynb`).

# Part A — Baseline grid sweep (4×3, full statistics)

Small, dense rectangular grid over `N_values × a_values`, enough trials per cell to get solid
box-plot statistics. This is the reference sweep; Part B below extends it to a much larger,
adaptive (N, a) grid.

---
## 1. Setup and imports

In [ ]:
import sys
import os
import numpy as np
import matplotlib.pyplot as plt

try:
    from google.colab import drive
    drive.mount('/content/drive')
    utils_path = "/content/drive/MyDrive/ColabProject/utils"
    print("Environment: Google Colab")
except ImportError:
    utils_path = os.path.abspath(os.path.join(os.getcwd(), "..", "utils"))
    print("Environment: Local")

if utils_path not in sys.path:
    sys.path.append(utils_path)

print(f"Utils path: {utils_path}")


Environment: Local
Utils path: <project_root>/utils


In [ ]:
from pipeline_utils import generate_percolating_network, distance_dependent_impedance
from network_visualization import plot_network_colored_by_resistance

print("Imports OK")


Imports OK


---
## 2. Experiment tracking

All figures and the full sweep data are saved to a new `experiments/exp_N/` folder (same `ExperimentManager` used across the project), so this sweep doesn't have to be re-run just to look at the numbers or plots again.

In [ ]:
from experiment_manager import setup_experiment

exp_manager = setup_experiment()


EXPERIMENT CONFIGURATION
  Environment: Local
  Experiment #: 1
  Timestamp: 20260810_122930
  Results directory: <project_root>/utils/../experiments/exp_1



---
## 3. Sweep configuration

For each (N, a) pair we keep drawing random networks until `n_trials` of them percolate (each draw that doesn't percolate is discarded, not counted as a trial). `max_attempts` caps the total number of draws per (N, a) pair as a safety valve for combinations far below the percolation threshold.

In [ ]:
N_values = [30, 45, 60, 75]
a_values = [0.4, 0.6, 0.8]
n_trials = 15
R0 = 1000.0
max_attempts = 400

print(f"Grid: {len(N_values)} x {len(a_values)} = {len(N_values) * len(a_values)} combinations, "
      f"{n_trials} percolating trials each")
print(f"N values: {N_values}")
print(f"a values: {a_values}")


Grid: 4 x 3 = 12 combinations, 15 percolating trials each
N values: [30, 45, 60, 75]
a values: [0.4, 0.6, 0.8]


---
## 4. Run the sweep

For each (N, a): draw networks (uniform-k pipeline under the hood, since that's what percolation/pruning is built on) until `n_trials` percolate, then recompute each one's impedance with the distance-dependent model. Also tracks the percolation success rate -- how many of the raw draws actually percolated -- since that's expected to drop sharply for small N*a^2 (near/below the stick-percolation threshold).

In [ ]:
results_grid = {}     # (N, a) -> list of R_eff (distance-dependent), one per percolating trial
perc_rate_grid = {}   # (N, a) -> fraction of raw draws that percolated
sample_networks = {}  # (N, a) -> first percolating network's full result dict (for plotting)

for N in N_values:
    for a in a_values:
        seed_base = (hash((N, round(a, 3))) % 100000)
        np.random.seed(seed_base)

        R_effs = []
        total_attempts = 0
        while len(R_effs) < n_trials:
            res, attempts = generate_percolating_network(N, a, k_uniform=R0, max_attempts=max_attempts)
            total_attempts += attempts
            if res is None:
                print(f"  [WARNING] N={N} a={a}: gave up after {total_attempts} draws, "
                      f"only found {len(R_effs)}/{n_trials} percolating networks")
                break
            if (N, a) not in sample_networks:
                sample_networks[(N, a)] = res
            rl = distance_dependent_impedance(res, R0)
            R_effs.append(rl['impedance'])

        rate = len(R_effs) / total_attempts if total_attempts else 0.0
        results_grid[(N, a)] = R_effs
        perc_rate_grid[(N, a)] = rate
        print(f"N={N:3d}  a={a:.1f}   {len(R_effs):2d}/{n_trials} percolating trials   "
              f"({total_attempts} draws, {rate:.1%} success rate)   "
              f"R_eff: mean={np.mean(R_effs):8.1f}  median={np.median(R_effs):8.1f}  ohm")

print("\nSweep complete.")


N= 30  a=0.4   15/15 percolating trials   (139 draws, 10.8% success rate)   R_eff: mean=   648.8  median=   547.6  ohm


N= 30  a=0.6   15/15 percolating trials   (73 draws, 20.5% success rate)   R_eff: mean=   316.1  median=   309.4  ohm


N= 30  a=0.8   15/15 percolating trials   (30 draws, 50.0% success rate)   R_eff: mean=   188.2  median=   191.9  ohm


N= 45  a=0.4   15/15 percolating trials   (41 draws, 36.6% success rate)   R_eff: mean=   410.7  median=   363.3  ohm


N= 45  a=0.6   15/15 percolating trials   (22 draws, 68.2% success rate)   R_eff: mean=   177.7  median=   169.5  ohm


...

N= 60  a=0.6   15/15 percolating trials   (22 draws, 68.2% success rate)   R_eff: mean=   129.9  median=   118.7  ohm


N= 60  a=0.8   15/15 percolating trials   (17 draws, 88.2% success rate)   R_eff: mean=    88.5  median=    84.6  ohm


N= 75  a=0.4   15/15 percolating trials   (19 draws, 78.9% success rate)   R_eff: mean=   214.2  median=   177.5  ohm


N= 75  a=0.6   15/15 percolating trials   (16 draws, 93.8% success rate)   R_eff: mean=   103.1  median=    98.1  ohm


N= 75  a=0.8   15/15 percolating trials   (15 draws, 100.0% success rate)   R_eff: mean=    62.8  median=    60.4  ohm

Sweep complete.


---
## 5. Summary table

In [ ]:
header = f"{'N':>4} {'a':>5} {'n':>4} {'mean(ohm)':>11} {'median(ohm)':>13} {'std(ohm)':>10} {'perc.rate':>10}"
print(header)
print("-" * len(header))
for N in N_values:
    for a in a_values:
        vals = np.array(results_grid[(N, a)])
        rate = perc_rate_grid[(N, a)]
        print(f"{N:4d} {a:5.2f} {len(vals):4d} {vals.mean():11.1f} {np.median(vals):13.1f} "
              f"{vals.std():10.1f} {rate:10.1%}")


   N     a    n   mean(ohm)   median(ohm)   std(ohm)  perc.rate
---------------------------------------------------------------
  30  0.40   15       648.8         547.6      218.8      10.8%
  30  0.60   15       316.1         309.4      110.3      20.5%
  30  0.80   15       188.2         191.9       44.5      50.0%
  45  0.40   15       410.7         363.3      179.9      36.6%
  45  0.60   15       177.7         169.5       46.3      68.2%
  45  0.80   15       131.0         124.0       32.9      62.5%
  60  0.40   15       255.7         252.4       66.4      71.4%
  60  0.60   15       129.9         118.7       28.5      68.2%
  60  0.80   15        88.5          84.6       25.0      88.2%
  75  0.40   15       214.2         177.5       87.8      78.9%
  75  0.60   15       103.1          98.1       21.0      93.8%
  75  0.80   15        62.8          60.4       11.7     100.0%


> **Note on local file paths (read before trusting `exp_1`/`exp_2` below):** this run of the notebook was executed on the Apuana cluster (CIn/UFPE), where `experiments/exp_1` and `experiments/exp_2` were genuinely the next free experiment numbers -- the cluster's `experiments/` folder was empty at the time (unrelated to this project's local numbering, which was already past `exp_800`). After syncing the results back, those two folders were manually moved to `experiments/exp_813/` (Part A) and `experiments/exp_814/` (Part B) locally, to avoid colliding with this project's own pre-existing `exp_1`/`exp_2` (unrelated experiments from March). The `Results directory: .../exp_1`, `.../exp_2` lines printed in the cell outputs below are the historical record of what actually ran on the cluster -- they are correct as printed, but no longer match where the files live locally.

---
## 6. Box plots: R_eff vs N, grouped by a

Log-scale y-axis, since R_eff spans more than an order of magnitude across the grid.

In [ ]:
fig, axes = plt.subplots(1, len(a_values), figsize=(5 * len(a_values), 5), sharey=True)

for ax, a in zip(axes, a_values):
    data = [results_grid[(N, a)] for N in N_values]
    ax.boxplot(data, positions=range(len(N_values)), widths=0.6)
    ax.set_xticks(range(len(N_values)))
    ax.set_xticklabels(N_values)
    ax.set_yscale('log')
    ax.set_xlabel('N (number of sticks)')
    ax.set_title(f'a = {a}')
    ax.grid(True, alpha=0.3, which='both')

axes[0].set_ylabel('R_eff (ohm, distance-dependent model, log scale)')
fig.suptitle(f'Effective resistance vs N, grouped by a  ({n_trials} percolating trials each)')
plt.tight_layout()
plt.show()
exp_manager.save_figure(fig, "box_R_eff_vs_N_by_a.png")


  [OK] Figure saved: box_R_eff_vs_N_by_a.png


'<project_root>/utils/../experiments/exp_1/box_R_eff_vs_N_by_a.png'

---
## 7. Box plots: R_eff vs a, grouped by N

In [ ]:
fig, axes = plt.subplots(1, len(N_values), figsize=(4.5 * len(N_values), 5), sharey=True)

for ax, N in zip(axes, N_values):
    data = [results_grid[(N, a)] for a in a_values]
    ax.boxplot(data, positions=range(len(a_values)), widths=0.6)
    ax.set_xticks(range(len(a_values)))
    ax.set_xticklabels(a_values)
    ax.set_yscale('log')
    ax.set_xlabel('a (aspect ratio)')
    ax.set_title(f'N = {N}')
    ax.grid(True, alpha=0.3, which='both')

axes[0].set_ylabel('R_eff (ohm, log scale)')
fig.suptitle(f'Effective resistance vs a, grouped by N  ({n_trials} percolating trials each)')
plt.tight_layout()
plt.show()
exp_manager.save_figure(fig, "box_R_eff_vs_a_by_N.png")


  [OK] Figure saved: box_R_eff_vs_a_by_N.png


'<project_root>/utils/../experiments/exp_1/box_R_eff_vs_a_by_N.png'

---
## 8. R_eff(N, a) and percolation rate as heatmaps

A compact view of the same sweep: mean $R_{eff}$ (log scale) and the raw percolation success rate, both as a function of (N, a).

In [ ]:
mean_matrix = np.array([[np.mean(results_grid[(N, a)]) for a in a_values] for N in N_values])
rate_matrix = np.array([[perc_rate_grid[(N, a)] for a in a_values] for N in N_values])

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11, 4.5))

im1 = ax1.imshow(np.log10(mean_matrix), cmap='viridis_r', aspect='auto')
ax1.set_xticks(range(len(a_values))); ax1.set_xticklabels(a_values)
ax1.set_yticks(range(len(N_values))); ax1.set_yticklabels(N_values)
ax1.set_xlabel('a'); ax1.set_ylabel('N')
ax1.set_title('Mean R_eff (log10 ohm)')
log_mean = np.log10(mean_matrix)
log_mid = (log_mean.max() + log_mean.min()) / 2
for i in range(len(N_values)):
    for j in range(len(a_values)):
        color = 'white' if log_mean[i, j] > log_mid else 'black'
        ax1.text(j, i, f"{mean_matrix[i, j]:.0f}", ha='center', va='center',
                 color=color, fontsize=9)
fig.colorbar(im1, ax=ax1, label='log10(R_eff / ohm)')

im2 = ax2.imshow(rate_matrix, cmap='viridis', vmin=0, vmax=1, aspect='auto')
ax2.set_xticks(range(len(a_values))); ax2.set_xticklabels(a_values)
ax2.set_yticks(range(len(N_values))); ax2.set_yticklabels(N_values)
ax2.set_xlabel('a'); ax2.set_ylabel('N')
ax2.set_title('Percolation success rate')
for i in range(len(N_values)):
    for j in range(len(a_values)):
        color = 'white' if rate_matrix[i, j] < 0.6 else 'black'
        ax2.text(j, i, f"{rate_matrix[i, j]:.2f}", ha='center', va='center', color=color, fontsize=9)
fig.colorbar(im2, ax=ax2, label='fraction of draws that percolated')

plt.tight_layout()
plt.show()
exp_manager.save_figure(fig, "heatmaps_mean_R_eff_and_perc_rate.png")


  [OK] Figure saved: heatmaps_mean_R_eff_and_perc_rate.png


'<project_root>/utils/../experiments/exp_1/heatmaps_mean_R_eff_and_perc_rate.png'

---
## 9. Representative network per (N, a)

One percolating network per (N, a) combination -- the first one found during the sweep above -- plotted with edges colored by their distance-dependent resistance. Useful as a visual reference for what these networks actually look like at each point of the grid (e.g. sparse/near-threshold N=30,a=0.4 vs dense N=75,a=0.8).

In [ ]:
for N in N_values:
    for a in a_values:
        res = sample_networks[(N, a)]
        rl = distance_dependent_impedance(res, R0)
        fig = plot_network_colored_by_resistance(
            res['merged_intersections'], res['merged_segment_to_nodes'],
            rl['edge_resistances'],
            title=f"N={N}, a={a}  ->  R_eff = {rl['impedance']:.0f} \u03a9 (distance-dependent)"
        )
        plt.show()
        exp_manager.save_figure(fig, f"network_N{N}_a{a}.png")
        exp_manager.save_network(N, a, res['points_1'], res['points_2'], res['angles'],
                                 filename=f"network_N{N}_a{a}.json")


Output hidden; open in https://colab.research.google.com to view.

---
## 10. Notes

- $R_{eff}$ decreases monotonically with both N and a, as expected: more sticks and/or longer sticks create more (and shorter) parallel conductive paths between source and drain.
- The percolation success rate drops sharply for small $N \cdot a^2$ (below the known 2D stick-percolation threshold, $N \cdot a^2 \approx 5.6$ for the excluded-area criterion) -- e.g. N=30, a=0.4 gives $N a^2 = 4.8$, and only a small fraction of raw draws percolate at all. That's expected, not a bug: near/below threshold, percolation is rare and the R_eff values we do get come from atypical, close-to-minimal conducting paths.
- `max_attempts` is a per-draw-batch cap; if a combination is far enough below threshold it can print a `[WARNING]` and return fewer than `n_trials` samples for that box.
- `N_values`/`a_values`/`n_trials` are all easy to widen for a finer sweep -- this grid (4 x 3 x 15 = 180 percolating networks) runs in well under a minute.

---
## 11. Save consolidated summary

In [ ]:
sweep_records = []
for N in N_values:
    for a in a_values:
        vals = results_grid[(N, a)]
        sweep_records.append({
            "N": N, "a": a, "n": len(vals),
            "mean_ohm": float(np.mean(vals)), "median_ohm": float(np.median(vals)),
            "std_ohm": float(np.std(vals)), "perc_rate": perc_rate_grid[(N, a)],
            "R_eff_values": vals,
        })

summary = {
    "experiment": exp_manager.get_info(),
    "config": {"N_values": N_values, "a_values": a_values, "n_trials": n_trials,
               "R0": R0, "max_attempts": max_attempts},
    "sweep": sweep_records,
}

exp_manager.save_json(summary, "effective_resistance_sweep_summary.json")

print(f"\nAll results saved to: {exp_manager.experiment_dir}")
for f in sorted(os.listdir(exp_manager.experiment_dir)):
    print(f"  - {f}")


  [OK] JSON saved: effective_resistance_sweep_summary.json

All results saved to: <project_root>/utils/../experiments/exp_1
  - box_R_eff_vs_N_by_a.png
  - box_R_eff_vs_a_by_N.png
  - effective_resistance_sweep_summary.json
  - heatmaps_mean_R_eff_and_perc_rate.png
  - network_N30_a0.4.json
  - network_N30_a0.4.png
  - network_N30_a0.6.json
  - network_N30_a0.6.png
  - network_N30_a0.8.json
  - network_N30_a0.8.png
  - network_N45_a0.4.json
  - network_N45_a0.4.png
  - network_N45_a0.6.json
  - network_N45_a0.6.png
  - network_N45_a0.8.json
  - network_N45_a0.8.png
  - network_N60_a0.4.json
  - network_N60_a0.4.png
  - network_N60_a0.6.json
  - network_N60_a0.6.png
  - network_N60_a0.8.json
  - network_N60_a0.8.png
  - network_N75_a0.4.json
  - network_N75_a0.4.png
  - network_N75_a0.6.json
  - network_N75_a0.6.png
  - network_N75_a0.8.json
  - network_N75_a0.8.png


# Part B — Extended adaptive-grid sweep (full version, meant for the cluster)

## 12. Sweep configuration (adaptive grid)

The 4x3 grid above is cheap because it never combines large N with large a. But the impedance
solve is a **dense** linear solve on the reduced Laplacian (`np.linalg.solve`), so its cost is
O(n_nodes^3), and n_nodes itself grows like N^2 * a^2 (more/longer sticks -> quadratically more
intersections). Benchmarking the corner directly:

| N | a | nodes | time/draw |
|---|---|---|---|
| 110 | 0.6 | 2177 | 2.7s |
| 110 | 1.0 | 4458 | 20.1s |
| 150 | 0.8 | 6430 | 48.5s |
| 150 | 1.0 | 8235 | **97s** |

So a naive rectangular grid up to N=150, a=1.0 would spend most of its time -- likely hours --
in a handful of cells in that corner, since dense networks also percolate on nearly every draw
(so all `n_trials` draws pay the full cost).

We use an **adaptive grid**: for each `a`, cap `N` so that the estimated node count
(`~0.5 * N^2 * a^2`) stays under a fixed budget. This keeps per-draw cost roughly bounded across
the whole grid while still covering N up to 150 (for small a) and a up to 1.0 (for smaller N).

This is now set to the **full version** -- 10 `a` values x 10 `N` per `a` = ~100 pairs, 30
percolating trials per pair (up to ~3000 percolating networks) -- meant to be run on the
cluster rather than locally. The original 5-value/4-per-a/10-trial preview is noted below
(commented out) as the config to fall back to for a quick local sanity check.

In [ ]:
import time

def n_max_for_a(a, node_cap=3000, N_ceiling=150, N_floor=20, round_to=5):
    """Largest N such that estimated node count (~0.5*N^2*a^2) stays under node_cap."""
    raw = np.sqrt(node_cap / (0.5 * a**2))
    capped = min(N_ceiling, raw)
    return int(max(N_floor, round(capped / round_to) * round_to))

# Full version, meant for the cluster: 10 a values x 10 N per a = ~100 pairs.
# Local quick-preview fallback (~20 pairs): a_values_ext = [0.2, 0.4, 0.6, 0.8, 1.0]; n_per_a = 4
a_values_ext = [0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9, 1.0, 1.1]
N_floor_ext = 20
n_per_a = 10

grid_ext = {}
pairs_ext = []
for a in a_values_ext:
    n_max = n_max_for_a(a)
    Ns = sorted(set(int(round(x / 5) * 5) for x in np.linspace(N_floor_ext, n_max, n_per_a)))
    grid_ext[a] = Ns
    pairs_ext.extend((N, a) for N in Ns)

print(f"{len(pairs_ext)} (N, a) pairs (adaptive grid, node count capped so the dense "
      f"impedance solve stays tractable)")
for a in a_values_ext:
    print(f"  a={a:.1f}: N_max={n_max_for_a(a):4d}   N in {grid_ext[a]}")

100 (N, a) pairs (adaptive grid, node count capped so the dense impedance solve stays tractable)
  a=0.2: N_max= 150   N in [20, 35, 50, 65, 80, 90, 105, 120, 135, 150]
  a=0.3: N_max= 150   N in [20, 35, 50, 65, 80, 90, 105, 120, 135, 150]
  a=0.4: N_max= 150   N in [20, 35, 50, 65, 80, 90, 105, 120, 135, 150]
  a=0.5: N_max= 150   N in [20, 35, 50, 65, 80, 90, 105, 120, 135, 150]
  a=0.6: N_max= 130   N in [20, 30, 45, 55, 70, 80, 95, 105, 120, 130]
  a=0.7: N_max= 110   N in [20, 30, 40, 50, 60, 70, 80, 90, 100, 110]
  a=0.8: N_max=  95   N in [20, 30, 35, 45, 55, 60, 70, 80, 85, 95]
  a=0.9: N_max=  85   N in [20, 25, 35, 40, 50, 55, 65, 70, 80, 85]
  a=1.0: N_max=  75   N in [20, 25, 30, 40, 45, 50, 55, 65, 70, 75]
  a=1.1: N_max=  70   N in [20, 25, 30, 35, 40, 50, 55, 60, 65, 70]


In [ ]:
n_trials_ext = 30
max_attempts_ext = 100

exp_manager_ext = setup_experiment()

print(f"\n{len(pairs_ext)} pairs x {n_trials_ext} trials each = up to "
      f"{len(pairs_ext) * n_trials_ext} percolating networks")

EXPERIMENT CONFIGURATION
  Environment: Local
  Experiment #: 2
  Timestamp: 20260810_123156
  Results directory: <project_root>/utils/../experiments/exp_2


100 pairs x 30 trials each = up to 3000 percolating networks


---
## 13. Run the extended sweep

Same logic as section 4, but over the adaptive grid from section 12. Checkpoints after every
(N, a) pair, so an interrupt doesn't lose the whole sweep.

In [ ]:
results_grid_ext = {}
perc_rate_grid_ext = {}

t_start = time.time()
for idx, (N, a) in enumerate(pairs_ext, start=1):
    seed_base = (hash((N, round(a, 3))) % 100000)
    np.random.seed(seed_base)

    R_effs = []
    total_attempts = 0
    while len(R_effs) < n_trials_ext:
        res, attempts = generate_percolating_network(N, a, k_uniform=R0, max_attempts=max_attempts_ext)
        total_attempts += attempts
        if res is None:
            print(f"  [WARNING] N={N} a={a}: gave up after {total_attempts} draws, "
                  f"only found {len(R_effs)}/{n_trials_ext} percolating networks")
            break
        rl = distance_dependent_impedance(res, R0)
        R_effs.append(rl['impedance'])

    rate = len(R_effs) / total_attempts if total_attempts else 0.0
    results_grid_ext[(N, a)] = R_effs
    perc_rate_grid_ext[(N, a)] = rate
    elapsed = time.time() - t_start
    mean_str = f"{np.mean(R_effs):8.1f}" if R_effs else "     n/a"
    print(f"[{idx:3d}/{len(pairs_ext)}] N={N:3d}  a={a:.1f}   {len(R_effs):2d}/{n_trials_ext} percolating   "
          f"({total_attempts:4d} draws, {rate:5.1%} success)   R_eff mean={mean_str} ohm   "
          f"[{elapsed:7.1f}s elapsed]")

    # checkpoint after every pair so a crash/interrupt doesn't lose the whole sweep
    exp_manager_ext.save_json(
        {"pairs_done": idx, "pairs_total": len(pairs_ext),
         "sweep": [{"N": n, "a": a2, "R_eff_values": v, "perc_rate": perc_rate_grid_ext[(n, a2)]}
                   for (n, a2), v in results_grid_ext.items()]},
        "effective_resistance_sweep_ext_CHECKPOINT.json",
    )

print(f"\nExtended sweep complete in {time.time() - t_start:.1f}s.")

  [WARNING] N=20 a=0.2: gave up after 100 draws, only found 0/30 percolating networks
[  1/100] N= 20  a=0.2    0/30 percolating   ( 100 draws,  0.0% success)   R_eff mean=     n/a ohm   [    0.2s elapsed]


  [OK] JSON saved: effective_resistance_sweep_ext_CHECKPOINT.json


  [WARNING] N=35 a=0.2: gave up after 100 draws, only found 0/30 percolating networks
[  2/100] N= 35  a=0.2    0/30 percolating   ( 100 draws,  0.0% success)   R_eff mean=     n/a ohm   [    0.7s elapsed]
  [OK] JSON saved: effective_resistance_sweep_ext_CHECKPOINT.json


  [WARNING] N=50 a=0.2: gave up after 100 draws, only found 0/30 percolating networks
[  3/100] N= 50  a=0.2    0/30 percolating   ( 100 draws,  0.0% success)   R_eff mean=     n/a ohm   [    1.7s elapsed]
  [OK] JSON saved: effective_resistance_sweep_ext_CHECKPOINT.json


  [WARNING] N=65 a=0.2: gave up after 100 draws, only found 0/30 percolating networks
[  4/100] N= 65  a=0.2    0/30 percolating   ( 100 draws,  0.0% success)   R_eff mean=     n/a ohm   [    3.4s elapsed]
  [OK] JSON saved: effective_resistance_sweep_ext_CHECKPOINT.json


...

[ 96/100] N= 50  a=1.1   30/30 percolating   (  30 draws, 100.0% success)   R_eff mean=   104.8 ohm   [ 6772.3s elapsed]
  [OK] JSON saved: effective_resistance_sweep_ext_CHECKPOINT.json


[ 97/100] N= 55  a=1.1   30/30 percolating   (  30 draws, 100.0% success)   R_eff mean=    91.4 ohm   [ 6833.0s elapsed]
  [OK] JSON saved: effective_resistance_sweep_ext_CHECKPOINT.json


[ 98/100] N= 60  a=1.1   30/30 percolating   (  30 draws, 100.0% success)   R_eff mean=    87.5 ohm   [ 6922.8s elapsed]
  [OK] JSON saved: effective_resistance_sweep_ext_CHECKPOINT.json


[ 99/100] N= 65  a=1.1   30/30 percolating   (  30 draws, 100.0% success)   R_eff mean=    72.7 ohm   [ 7058.3s elapsed]
  [OK] JSON saved: effective_resistance_sweep_ext_CHECKPOINT.json


[100/100] N= 70  a=1.1   30/30 percolating   (  30 draws, 100.0% success)   R_eff mean=    59.4 ohm   [ 7260.3s elapsed]
  [OK] JSON saved: effective_resistance_sweep_ext_CHECKPOINT.json

Extended sweep complete in 7260.4s.


---
## 14. Box plots: R_eff vs N, grouped by a (ragged grid)

Same idea as section 6, but each `a` now has its own N range (the adaptive grid isn't a full
rectangle), so each panel gets its own x-axis.

In [ ]:
fig, axes = plt.subplots(2, 5, figsize=(22, 9))
axes = axes.flatten()

for ax, a in zip(axes, a_values_ext):
    Ns = grid_ext[a]
    # some (N, a) below the percolation threshold may have 0 trials -- skip those, not the box
    Ns_nonempty = [N for N in Ns if results_grid_ext[(N, a)]]
    if not Ns_nonempty:
        ax.text(0.5, 0.5, "no percolating\ntrials found", ha='center', va='center',
                transform=ax.transAxes, fontsize=10, color='gray')
        ax.set_title(f'a = {a:.1f}')
        continue
    data = [results_grid_ext[(N, a)] for N in Ns_nonempty]
    ax.boxplot(data, positions=range(len(Ns_nonempty)), widths=0.6)
    ax.set_xticks(range(len(Ns_nonempty)))
    ax.set_xticklabels(Ns_nonempty, rotation=45)
    ax.set_yscale('log')
    ax.set_xlabel('N (number of sticks)')
    ax.set_ylabel('R_eff (ohm, log scale)')
    ax.set_title(f'a = {a:.1f}')
    ax.grid(True, alpha=0.3, which='both')

fig.suptitle(f'Extended sweep: R_eff vs N, grouped by a  '
             f'({n_trials_ext} percolating trials each, adaptive grid, {len(pairs_ext)} pairs)')
plt.tight_layout()
plt.show()
exp_manager_ext.save_figure(fig, "box_R_eff_vs_N_by_a_ext.png")

  [OK] Figure saved: box_R_eff_vs_N_by_a_ext.png


'<project_root>/utils/../experiments/exp_2/box_R_eff_vs_N_by_a_ext.png'

---
## 15. R_eff(N, a) and percolation rate over the adaptive grid

The grid isn't a rectangle anymore, so a matrix heatmap doesn't apply -- these are scatter plots
over the actual (N, a) points sampled, colored by mean R_eff and by percolation success rate.

In [ ]:
Ns_all = np.array([N for (N, a) in pairs_ext])
as_all = np.array([a for (N, a) in pairs_ext])
means_all = np.array([np.mean(results_grid_ext[(N, a)]) if results_grid_ext[(N, a)] else np.nan
                       for (N, a) in pairs_ext])
rates_all = np.array([perc_rate_grid_ext[(N, a)] for (N, a) in pairs_ext])

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

sc1 = axes[0].scatter(Ns_all, as_all, c=np.log10(means_all), cmap='viridis_r',
                       s=80, edgecolor='k', linewidth=0.3)
axes[0].set_xlabel('N'); axes[0].set_ylabel('a')
axes[0].set_title('Mean R_eff (log10 ohm)')
fig.colorbar(sc1, ax=axes[0], label='log10(R_eff / ohm)')

sc2 = axes[1].scatter(Ns_all, as_all, c=rates_all, cmap='viridis', vmin=0, vmax=1,
                       s=80, edgecolor='k', linewidth=0.3)
axes[1].set_xlabel('N'); axes[1].set_ylabel('a')
axes[1].set_title('Percolation success rate')
fig.colorbar(sc2, ax=axes[1], label='fraction of draws that percolated')

fig.suptitle(f'Extended sweep over {len(pairs_ext)} (N, a) pairs')
plt.tight_layout()
plt.show()
exp_manager_ext.save_figure(fig, "scatter_R_eff_and_perc_rate_ext.png")

  [OK] Figure saved: scatter_R_eff_and_perc_rate_ext.png


'<project_root>/utils/../experiments/exp_2/scatter_R_eff_and_perc_rate_ext.png'

---
## 16. R_eff collapse vs N*a^2

With ~20 points spread over N and a independently, we can check whether R_eff is really a
function of the combined variable $N \cdot a^2$ (the stick-percolation excluded-area parameter),
as the notes in section 10 above assumed from just 12 points.

In [ ]:
na2 = Ns_all * as_all**2
valid = ~np.isnan(means_all)

fig, ax = plt.subplots(figsize=(7.5, 5.5))
sc = ax.scatter(na2[valid], means_all[valid], c=as_all[valid], cmap='plasma',
                 s=45, edgecolor='k', linewidth=0.3)
ax.set_xscale('log'); ax.set_yscale('log')
ax.set_xlabel('N * a^2'); ax.set_ylabel('mean R_eff (ohm)')
ax.set_title('R_eff vs N*a^2 (color = a)')
ax.grid(True, alpha=0.3, which='both')
fig.colorbar(sc, ax=ax, label='a')
plt.tight_layout()
plt.show()
exp_manager_ext.save_figure(fig, "scatter_R_eff_vs_Na2_ext.png")

  [OK] Figure saved: scatter_R_eff_vs_Na2_ext.png


'<project_root>/utils/../experiments/exp_2/scatter_R_eff_vs_Na2_ext.png'

---
## 17. Save consolidated summary

In [ ]:
sweep_records_ext = []
for (N, a) in pairs_ext:
    vals = results_grid_ext[(N, a)]
    sweep_records_ext.append({
        "N": N, "a": a, "n": len(vals),
        "mean_ohm": float(np.mean(vals)) if vals else None,
        "median_ohm": float(np.median(vals)) if vals else None,
        "std_ohm": float(np.std(vals)) if vals else None,
        "perc_rate": perc_rate_grid_ext[(N, a)],
        "R_eff_values": vals,
    })

summary_ext = {
    "experiment": exp_manager_ext.get_info(),
    "config": {
        "pairs": pairs_ext, "n_trials": n_trials_ext, "R0": R0,
        "max_attempts": max_attempts_ext,
        "grid_design": "adaptive: N_max(a) chosen so estimated node count (~0.5*N^2*a^2) stays <= node_cap=3000",
    },
    "sweep": sweep_records_ext,
}

exp_manager_ext.save_json(summary_ext, "effective_resistance_sweep_summary_ext.json")

print(f"\nAll results saved to: {exp_manager_ext.experiment_dir}")
for f in sorted(os.listdir(exp_manager_ext.experiment_dir)):
    print(f"  - {f}")

  [OK] JSON saved: effective_resistance_sweep_summary_ext.json

All results saved to: <project_root>/utils/../experiments/exp_2
  - box_R_eff_vs_N_by_a_ext.png
  - effective_resistance_sweep_ext_CHECKPOINT.json
  - effective_resistance_sweep_summary_ext.json
  - scatter_R_eff_and_perc_rate_ext.png
  - scatter_R_eff_vs_Na2_ext.png
